In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
#IMPORT STATEMENTS
import os
import csv
import copy
import glob
import json
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

os.chdir('/kaggle/working')
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# CONFIGURATION
DATA_DIR = '/kaggle/input/datasets/ameerutheenahamedt/feature-engineered-dataset'
PARQUET_PATH = None
WORK_DIR = '/kaggle/working'

TARGET = 'Modal_Price'
GROUP_COLS = None

SEQ_LEN = 7
TRAIN_END_YEAR = 2024
VAL_YEAR = 2025
TEST_YEAR = 2026

HIDDEN_SIZE = 64
NUM_LAYERS = 2
BATCH_SIZE = 32
EPOCHS = 30
LR = 0.001

In [ ]:
# Load dataset

parquet_files = glob.glob(os.path.join(DATA_DIR, '*.parquet'))

print("Parquet files found:", len(parquet_files))

if not parquet_files:
    raise FileNotFoundError(f"No Parquet files found inside {DATA_DIR}")

dataframes = []

for file in parquet_files:
    print("Loading:", os.path.basename(file))
    temp_df = pd.read_parquet(file)
    dataframes.append(temp_df)

df = pd.concat(dataframes, ignore_index=True)

assert TARGET in df.columns, f"Target column '{TARGET}' not found. Columns: {df.columns.tolist()}"

df = df.dropna(subset=[TARGET]).reset_index(drop=True)

print("\nDataset loaded successfully")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

In [ ]:
# Inspect the real columns
print(df.dtypes)
print("\nMissing values (top 15):")
print(df.isna().sum().sort_values(ascending=False).head(15))
print("\nTarget summary:")
print(df[TARGET].describe())

In [ ]:
# Detect the date, sort chronologically, extract the year
date_col = None
for c in df.columns:
    if 'date' in c.lower() and not pd.api.types.is_numeric_dtype(df[c]):
        parsed = pd.to_datetime(df[c], errors='coerce')
        if parsed.notna().mean() > 0.9:
            df[c] = parsed
            date_col = c
            break

if date_col is None:
    lower = {c.lower(): c for c in df.columns}
    if 'year' in lower and 'month' in lower:
        day = df[lower['day']] if 'day' in lower else 1
        df['Date'] = pd.to_datetime(dict(year=df[lower['year']], month=df[lower['month']], day=day), errors='coerce')
        date_col = 'Date'
    else:
        raise ValueError(f"No date column or Year/Month columns found. Columns: {df.columns.tolist()}")

df = df.dropna(subset=[date_col])
df = df.sort_values(date_col, kind='mergesort').reset_index(drop=True)
df['_year'] = df[date_col].dt.year

print("Date column:", date_col)
print("Date range:", df[date_col].min(), "->", df[date_col].max())
print("\nRows per year:")
print(df['_year'].value_counts().sort_index())

In [ ]:
# Detect grouping columns (commodity / market / state ...) and numeric feature columns
GROUP_KEYWORDS = ['commodity', 'market', 'state', 'district', 'variety', 'crop']
EXCLUDE_WORDS = ['price', 'lag', 'roll', 'mean', 'avg', 'std', 'min', 'max', 'diff', 'pct', 'ma_', 'ema', 'trend', 'season', 'volatil']

if GROUP_COLS is None:
    GROUP_COLS = []
    for c in df.columns:
        if c in (TARGET, date_col, '_year'):
            continue
        name = c.lower()
        if not any(k in name for k in GROUP_KEYWORDS):
            continue
        if any(w in name for w in EXCLUDE_WORDS):
            continue
        if pd.api.types.is_float_dtype(df[c]):
            continue
        GROUP_COLS.append(c)

print("Group columns:", GROUP_COLS)
if GROUP_COLS:
    print("Number of groups:", df.groupby(GROUP_COLS, sort=False, dropna=False).ngroup().nunique())

feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != '_year']
print("\nNumber of input features:", len(feature_cols))
print("Feature columns:", feature_cols)

In [ ]:
# Scaling: fit ONLY on training-period rows, then apply to everything
train_row_mask = (df['_year'] <= TRAIN_END_YEAR).to_numpy()
assert train_row_mask.sum() > 0, "No training rows found (year <= TRAIN_END_YEAR)"

df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)
train_medians = df.loc[train_row_mask, feature_cols].median()
df[feature_cols] = df[feature_cols].fillna(train_medians).fillna(0)

feature_scaler = StandardScaler().fit(df.loc[train_row_mask, feature_cols])
target_scaler = StandardScaler().fit(df.loc[train_row_mask, [TARGET]])

X_all = feature_scaler.transform(df[feature_cols]).astype(np.float32)
y_all = target_scaler.transform(df[[TARGET]]).astype(np.float32).ravel()

y_mean = float(target_scaler.mean_[0])
y_std = float(target_scaler.scale_[0])

joblib.dump(feature_scaler, os.path.join(WORK_DIR, 'feature_scaler.joblib'))
joblib.dump(target_scaler, os.path.join(WORK_DIR, 'target_scaler.joblib'))

print("Training rows used to fit scalers:", int(train_row_mask.sum()))
print(f"Target mean: {y_mean:.2f} | Target std: {y_std:.2f}")
print("Saved: feature_scaler.joblib, target_scaler.joblib")

In [ ]:
# Build sequences per group. A window = the SEQ_LEN rows BEFORE the target row (past only).
if GROUP_COLS:
    gid = df.groupby(GROUP_COLS, sort=False, dropna=False).ngroup().to_numpy()
else:
    gid = np.zeros(len(df), dtype=np.int64)

order = np.argsort(gid, kind='stable')
bounds = np.flatnonzero(np.diff(gid[order])) + 1
group_positions = np.split(order, bounds)

win_list, tgt_list = [], []

for pos in group_positions:
    if len(pos) <= SEQ_LEN:
        continue

    win_list.append(sliding_window_view(pos, SEQ_LEN)[:-1])
    tgt_list.append(pos[SEQ_LEN:])

win_idx = np.concatenate(win_list).astype(np.int64)
tgt_idx = np.concatenate(tgt_list).astype(np.int64)

# Chronological split by the YEAR OF THE TARGET ROW
tgt_year = df['_year'].to_numpy()[tgt_idx]

train_sel = tgt_year <= TRAIN_END_YEAR
val_sel = tgt_year == VAL_YEAR
test_sel = tgt_year == TEST_YEAR

assert train_sel.sum() > 0, "No training sequences"
assert val_sel.sum() > 0, f"No validation sequences for year {VAL_YEAR}"
assert test_sel.sum() > 0, f"No test sequences for year {TEST_YEAR}"

# Leakage checks
dates = df[date_col].to_numpy()

assert (dates[win_idx[:, -1]] <= dates[tgt_idx]).all(), \
    "A window contains future rows"

print("Latest train target date:", dates[tgt_idx[train_sel]].max())
print("Earliest val target date:", dates[tgt_idx[val_sel]].min())
print("Earliest test target date:", dates[tgt_idx[test_sel]].min())

train_win, train_tgt = win_idx[train_sel], tgt_idx[train_sel]
val_win, val_tgt = win_idx[val_sel], tgt_idx[val_sel]
test_win, test_tgt = win_idx[test_sel], tgt_idx[test_sel]

print(f"\nSequences -> Train: {len(train_tgt)} | Validation: {len(val_tgt)} | Test: {len(test_tgt)}")
print("Sequence shape per sample:", (SEQ_LEN, len(feature_cols)))

In [ ]:
# Create datasets and dataloaders
class PriceSequenceDataset(Dataset):
    def __init__(self, X, y, win_idx, tgt_idx):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
        self.win_idx = torch.from_numpy(win_idx)
        self.tgt_idx = torch.from_numpy(tgt_idx)

    def __len__(self):
        return len(self.tgt_idx)

    def __getitem__(self, i):
        return self.X[self.win_idx[i]], self.y[self.tgt_idx[i]]

train_set = PriceSequenceDataset(X_all, y_all, train_win, train_tgt)
val_set = PriceSequenceDataset(X_all, y_all, val_win, val_tgt)
test_set = PriceSequenceDataset(X_all, y_all, test_win, test_tgt)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

xb, yb = next(iter(train_loader))
print("Batch input shape:", tuple(xb.shape), "| Batch target shape:", tuple(yb.shape))

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

model = LSTMRegressor(
    input_size=len(feature_cols),
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS
)

model_config = {
    'input_size': len(feature_cols),
    'hidden_size': HIDDEN_SIZE,
    'num_layers': NUM_LAYERS,
    'seq_len': SEQ_LEN,
    'feature_cols': feature_cols,
    'group_cols': GROUP_COLS,
    'target': TARGET,
    'date_col': date_col,
    'y_mean': y_mean,
    'y_std': y_std,
}

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
print("Target column:", TARGET)
print("Number of input features:", len(feature_cols))
print("Sequence length:", SEQ_LEN)
print("Feature columns:", feature_cols)

In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Move the model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
print(next(model.parameters()).device)

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, device, y_std, config):
    since = time.time()
    best_val_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())
    best_epoch = 0

    # Initialize CSV logging
    csv_filename = '/kaggle/working/training_log.csv'
    with open(csv_filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Epoch', 'Train Loss', 'Validation Loss', 'Train RMSE', 'Validation RMSE'])

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        train_se = 0.0
        total = 0

        # Training loop
        for inputs, targets in train_loader:
            inputs = inputs.to(device).float()
            targets = targets.to(device).float()

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            train_se += (((outputs.detach() - targets) * y_std) ** 2).sum().item()   # error in original price units
            total += targets.size(0)

        avg_train_loss = running_loss / total
        train_rmse = (train_se / total) ** 0.5

        print(f"Epoch [{epoch+1}/{epochs}], Training Loss: {avg_train_loss:.4f}, Train RMSE: {train_rmse:.4f}")

        model.eval()
        val_loss = 0.0
        val_se = 0.0
        val_total = 0

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs = inputs.to(device).float()
                targets = targets.to(device).float()

                outputs = model(inputs)
                loss = criterion(outputs, targets)

                val_loss += loss.item() * inputs.size(0)
                val_se += (((outputs - targets) * y_std) ** 2).sum().item()
                val_total += targets.size(0)

        avg_val_loss = val_loss / val_total
        val_rmse = (val_se / val_total) ** 0.5

        print(f"Epoch [{epoch+1}/{epochs}], Val Loss: {avg_val_loss:.4f}, Val RMSE: {val_rmse:.4f}")

        # Append metrics to CSV
        with open(csv_filename, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([epoch+1, avg_train_loss, avg_val_loss, train_rmse, val_rmse])

        # Save the best model (lowest validation loss)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1

    elapsed_time = time.time() - since
    print(f"\nTraining complete in {elapsed_time // 60:.0f}m {elapsed_time % 60:.0f}s")

    torch.save({'model_state_dict': best_model_wts, 'config': config, 'best_epoch': best_epoch,
                'best_val_loss': best_val_loss}, "/kaggle/working/lstm_price_prediction.pth")
    print(f"\n✔️ Best model saved at epoch {best_epoch} with Validation Loss {best_val_loss:.4f}")
    print(f"Best Epoch: {best_epoch}")
    print(f"Best Validation Loss: {best_val_loss:.4f}")
    print(f"Total Training Time: {elapsed_time // 60:.0f}m {elapsed_time % 60:.0f}s")

    model.load_state_dict(best_model_wts)
    return model, best_epoch, best_val_loss, elapsed_time

In [ ]:
# Limit training sequences to a manageable size
MAX_TRAIN_SAMPLES = 500000

if len(train_tgt) > MAX_TRAIN_SAMPLES:
    rng = np.random.default_rng(42)
    selected = rng.choice(len(train_tgt), MAX_TRAIN_SAMPLES, replace=False)
    train_win = train_win[selected]
    train_tgt = train_tgt[selected]

print("Training sequences used:", len(train_tgt))
print("Validation sequences:", len(val_tgt))
print("Test sequences:", len(test_tgt))

In [ ]:
train_set = PriceSequenceDataset(X_all, y_all, train_win, train_tgt)
val_set = PriceSequenceDataset(X_all, y_all, val_win, val_tgt)
test_set = PriceSequenceDataset(X_all, y_all, test_win, test_tgt)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
model, best_epoch, best_val_loss, training_time = train_model(
    model, train_loader, val_loader, criterion, optimizer,
    epochs=EPOCHS, device=device, y_std=y_std, config=model_config
)

In [ ]:
# Evaluate the BEST model on the 2026 test set
model.eval()
test_loss = 0.0
all_preds = []
all_targets = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device).float()
        targets = targets.to(device).float()

        outputs = model(inputs)
        loss = criterion(outputs, targets)

        test_loss += loss.item() * inputs.size(0)
        all_preds.append(outputs.cpu())
        all_targets.append(targets.cpu())

avg_test_loss = test_loss / len(test_loader.dataset)

pred_scaled = torch.cat(all_preds).numpy()
true_scaled = torch.cat(all_targets).numpy()

# Back to the original price scale
y_pred = target_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
y_true = target_scaler.inverse_transform(true_scaled.reshape(-1, 1)).ravel()

test_rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
test_mae = float(mean_absolute_error(y_true, y_pred))
test_r2 = float(r2_score(y_true, y_pred))
test_mape = float(mean_absolute_percentage_error(y_true, y_pred) * 100)

print(f"Test Loss (MSE, scaled): {avg_test_loss:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE:  {test_mae:.4f}")
print(f"Test R2:   {test_r2:.4f}")
print(f"Test MAPE: {test_mape:.2f}%")

In [ ]:
# Save test predictions
meta_cols = [date_col] + list(GROUP_COLS)
test_results = df.iloc[test_tgt][meta_cols].reset_index(drop=True)
test_results['Actual_Modal_Price'] = y_true
test_results['Predicted_Modal_Price'] = y_pred

test_results.to_csv('/kaggle/working/test_predictions.csv', index=False)
print("Saved: test_predictions.csv | rows:", len(test_results))
test_results.head()

In [ ]:
# Graph 1: Training loss vs validation loss across all epochs
log = pd.read_csv('/kaggle/working/training_log.csv')

plt.figure(figsize=(8, 5))
plt.plot(log['Epoch'], log['Train Loss'], marker='o', markersize=3, label='Train Loss')
plt.plot(log['Epoch'], log['Validation Loss'], marker='o', markersize=3, label='Validation Loss')
plt.axvline(best_epoch, color='gray', linestyle='--', label=f'Best epoch ({best_epoch})')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss (scaled)')
plt.title('LSTM Training Loss vs Validation Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/loss_curve.png', dpi=150)
plt.show()

In [ ]:
# Graph 2: Actual price vs predicted price (2026 test set)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].scatter(y_true, y_pred, s=8, alpha=0.4)
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
ax[0].plot(lims, lims, 'r--', label='Perfect prediction')
ax[0].set_xlabel('Actual Modal_Price')
ax[0].set_ylabel('Predicted Modal_Price')
ax[0].set_title('Actual vs Predicted (Test)')
ax[0].legend()
ax[0].grid(alpha=0.3)

daily = test_results.groupby(date_col)[['Actual_Modal_Price', 'Predicted_Modal_Price']].mean().sort_index()
ax[1].plot(daily.index, daily['Actual_Modal_Price'], label='Actual')
ax[1].plot(daily.index, daily['Predicted_Modal_Price'], label='Predicted')
ax[1].set_xlabel('Date')
ax[1].set_ylabel('Average Modal_Price')
ax[1].set_title('Average Price over Time (Test)')
ax[1].legend()
ax[1].grid(alpha=0.3)
fig.autofmt_xdate()

plt.tight_layout()
plt.savefig('/kaggle/working/actual_vs_predicted.png', dpi=150)
plt.show()

In [ ]:
# Save preprocessing configuration (needed later for backend inference)
preprocessing_config = {
    'target': TARGET,
    'date_col': date_col,
    'group_cols': GROUP_COLS,
    'feature_cols': feature_cols,
    'seq_len': SEQ_LEN,
    'hidden_size': HIDDEN_SIZE,
    'num_layers': NUM_LAYERS,
    'train_end_year': TRAIN_END_YEAR,
    'val_year': VAL_YEAR,
    'test_year': TEST_YEAR,
    'y_mean': y_mean,
    'y_std': y_std,
    'train_medians': {k: float(v) for k, v in train_medians.items()},
}
with open('/kaggle/working/preprocessing_config.json', 'w') as f:
    json.dump(preprocessing_config, f, indent=2)
print("Saved: preprocessing_config.json")

In [ ]:
# Verify all required output files
required_files = [
    'lstm_price_prediction.pth',
    'training_log.csv',
    'test_predictions.csv',
    'feature_scaler.joblib',
    'target_scaler.joblib',
    'preprocessing_config.json',
    'loss_curve.png',
    'actual_vs_predicted.png',
]

all_ok = True
for fname in required_files:
    fpath = os.path.join('/kaggle/working', fname)
    if os.path.exists(fpath):
        print(f"OK       {fname} ({os.path.getsize(fpath):,} bytes)")
    else:
        print(f"MISSING  {fname}")
        all_ok = False

print("\nAll required files present:", all_ok)

In [ ]:
# Final summary
print("=" * 45)
print("Model: LSTM")
print(f"Epochs: {EPOCHS}")
print(f"Best Epoch: {best_epoch}")
print(f"Best Validation Loss: {best_val_loss:.4f}")
print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE: {test_mae:.4f}")
print(f"Test R²: {test_r2:.4f}")
print(f"Test MAPE: {test_mape:.2f}%")
print(f"Training Time: {training_time // 60:.0f}m {training_time % 60:.0f}s")
print("=" * 45)

In [ ]:
# Final Extended Evaluation: MAE, RMSE, MASE, sMAPE, WAPE
import numpy as np
import pandas as pd

# Use the actual LSTM predictions and actual target values from the test set
y_true = np.array(y_true, dtype=np.float64)
y_pred = np.array(y_pred, dtype=np.float64)

# 1. MAE
mae = float(np.mean(np.abs(y_true - y_pred)))

# 2. RMSE
rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

# 3. MASE (Mean Absolute Scaled Error)
scale = float(np.mean(np.abs(np.diff(y_true))))
mase = float(mae / scale) if scale > 0 else 0.0

# 4. sMAPE (Symmetric Mean Absolute Percentage Error)
denom = np.abs(y_true) + np.abs(y_pred)
valid = denom > 1e-8
smape = float(np.mean(2.0 * np.abs(y_true[valid] - y_pred[valid]) / denom[valid]) * 100.0)

# 5. WAPE (Weighted Absolute Percentage Error)
wape = float((np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))) * 100.0)

# Print exact requested format
print("Final LSTM Evaluation Results")
print(f"MAE   : {mae:.4f}")
print(f"RMSE  : {rmse:.4f}")
print(f"MASE  : {mase:.4f}")
print(f"sMAPE : {smape:.4f}%")
print(f"WAPE  : {wape:.4f}%")
print()
print(f"Best Epoch: {best_epoch}")
print(f"Best Validation Loss: {best_val_loss:.2E}")
log_df = pd.read_csv('/kaggle/working/training_log.csv')
best_val_rmse = float(log_df.loc[log_df['Epoch'] == best_epoch, 'Validation RMSE'].values[0])
print(f"Best Validation RMSE: {best_val_rmse:.4f}")
